# Lab 08 — How Models Learn: Gradient Descent, Algorithm Selection, and Dimensionality

In this lab you will implement a gradient descent update step, rank features by their predictive signal, and measure distance collapse in high-dimensional spaces.

No model training is required in Sections A and B — focus on the mechanics that every ML algorithm is built on.

**Concepts covered:** gradient descent, learning rate, No Free Lunch Theorem, feature importance, curse of dimensionality.

**Reference working sessions:**
- `working-sessions/ml_concepts/08_gradient_descent.ipynb`
- `working-sessions/ml_concepts/09_the_no_free_lunch_theorem.ipynb`
- `working-sessions/ml_concepts/10_feature_importance.ipynb`
- `working-sessions/ml_concepts/11_the_curse_of_dimensionality.ipynb`
- `working-sessions/math/calculus/06_gradient_and_gradient_descent.ipynb`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tkh_utils import (
    PALETTE, FONT, base_layout,
    check_answer, make_answer_key, make_grading_summary,
    load_california_housing,
)

_ak = make_answer_key({
    'q1': 'C',
    'q2': 'B',
    'q3': 'B',
    'q4': 'C',
})

---
## Section A — Multiple Choice

Fill in each answer variable with the letter of the best answer (A, B, C, or D).

In [ ]:
# Q1 — You are training a model and set the learning rate to 2.0.
# After a few steps you notice the loss is increasing instead of decreasing.
# What is most likely happening?
#
#   A) The model needs more data before the loss can decrease
#   B) The learning rate is too small — gradient descent is moving in tiny steps
#   C) The learning rate is too large — gradient descent is overshooting the minimum and diverging
#   D) The loss function has no minimum, so gradient descent cannot converge

q1_answer = "___"  # Replace with A, B, C, or D

assert q1_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q1_answer, _ak['q1']), "Not quite — revisit working-sessions/ml_concepts/08_gradient_descent.ipynb and observe what happens when the learning rate slider is set above 1.0"
print("\u2713 Question 1 correct!")

In [ ]:
# Q2 — The No Free Lunch Theorem states that:
#
#   A) Algorithms with more parameters always outperform simpler ones
#   B) No single algorithm is best for every problem — performance depends on the structure of the data
#   C) Adding more training data always improves any algorithm's performance
#   D) Complex models require exponentially more data to generalize

q2_answer = "___"  # Replace with A, B, C, or D

assert q2_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q2_answer, _ak['q2']), "Not quite — revisit working-sessions/ml_concepts/09_the_no_free_lunch_theorem.ipynb"
print("\u2713 Question 2 correct!")

In [ ]:
# Q3 — Tree-based feature importance tends to inflate the importance scores of:
#
#   A) Features with many missing values
#   B) Features with high cardinality — many unique values, such as IDs or timestamps
#   C) Features that are negatively correlated with the target
#   D) Features that appear only in the first few splits

q3_answer = "___"  # Replace with A, B, C, or D

assert q3_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q3_answer, _ak['q3']), "Not quite — revisit working-sessions/ml_concepts/10_feature_importance.ipynb and toggle the 'ID column' option"
print("\u2713 Question 3 correct!")

In [ ]:
# Q4 — As the number of dimensions grows, what happens to the ratio of
# the nearest to the farthest neighbor distance for a randomly chosen point?
#
#   A) The ratio grows, making the nearest neighbor much closer than the farthest
#   B) The ratio stays constant — high dimensions do not affect distance structure
#   C) The ratio collapses toward zero — all points become roughly equidistant
#   D) The ratio fluctuates unpredictably depending on the dataset

q4_answer = "___"  # Replace with A, B, C, or D

assert q4_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q4_answer, _ak['q4']), "Not quite — revisit working-sessions/ml_concepts/11_the_curse_of_dimensionality.ipynb"
print("\u2713 Question 4 correct!")

In [ ]:
make_grading_summary([
    (q1_answer, _ak['q1'], "Q1: Learning rate and gradient descent divergence"),
    (q2_answer, _ak['q2'], "Q2: The No Free Lunch Theorem"),
    (q3_answer, _ak['q3'], "Q3: Tree-based feature importance bias"),
    (q4_answer, _ak['q4'], "Q4: Distance collapse in high dimensions"),
], total=4)

---
## Section B — Coding Exercises

The exercises below use NumPy and a real dataset. No model training is required — focus on the underlying mechanics that make training work.

### B1 — Implement a gradient descent update step

Every ML optimizer — whether training a linear model, a neural network, or a gradient boosting tree — executes the same core operation on every step. The cell below sets up a simple quadratic loss function and asks you to implement that step.

In [ ]:
# B1 — Implement a gradient descent update step
#
# We have a simple quadratic loss: L(w) = (w - 3)^2
# The gradient is: dL/dw = 2 * (w - 3)
#
# At each step, gradient descent moves w in the direction that reduces the loss:
#   w_new = w_old - learning_rate * gradient
#
# Complete the function below.

def gradient_step(w, gradient, learning_rate):
    new_w = ___          # YOUR CODE — one line
    return new_w


# --- Run 5 gradient descent steps ---
w = 10.0          # starting far from the minimum at w = 3
learning_rate = 0.1
loss_history = []

for step in range(5):
    gradient = 2 * (w - 3)
    loss = (w - 3) ** 2
    loss_history.append(loss)
    w = gradient_step(w, gradient, learning_rate)

print(f"Loss at step 0 : {loss_history[0]:.4f}")
print(f"Loss at step 4 : {loss_history[-1]:.4f}")
print(f"Final w        : {w:.4f}  (minimum is at w = 3)")

# --- checks ---
assert callable(gradient_step), "gradient_step must be a function"
assert abs(gradient_step(10.0, 14.0, 0.1) - 8.6) < 1e-6, "Check your formula: new_w = w - learning_rate * gradient"
assert loss_history[-1] < loss_history[0], "Loss should decrease across steps — check that gradient_step returns w - lr * gradient"
assert w < 10.0, "w should have moved toward the minimum at 3.0"
print("\u2713 B1 complete!")

### B2 — Rank features by correlation with the target

Before fitting any model, data scientists rank features by how strongly they relate to the target. Pearson correlation is one of the fastest ways to do this for continuous targets.

In [ ]:
# B2 — Rank features by correlation with the target
#
# California Housing data: 8 features, target = median house value
# The target column in the combined dataframe is named 'MedHouseVal'

X, y = load_california_housing()

# Combine features and target into one dataframe for correlation analysis
full_df = X.copy()
full_df['MedHouseVal'] = y

features = X.columns.tolist()

# Compute Pearson correlation of each feature with the target
# Fill in the target column name and the sort direction
correlations = full_df[features].corrwith(full_df[___])      # YOUR CODE — fill in the target column name
correlations_ranked = correlations.sort_values(key=abs, ascending=___)   # YOUR CODE — True or False

top_feature = correlations_ranked.index[0]

print("Feature correlations with median house value (ranked by strength):")
for feat, corr in correlations_ranked.items():
    print(f"  {feat:<14} {corr:+.3f}")

print(f"\nStrongest predictor: {top_feature}")

# --- checks ---
assert isinstance(correlations_ranked, pd.Series), "correlations_ranked should be a pandas Series"
assert correlations_ranked.index[0] == 'MedInc', "Sort by absolute value descending — the strongest predictor should be 'MedInc'"
assert correlations_ranked.iloc[0] > 0, "MedInc should have a positive correlation with house values"
print("\u2713 B2 complete!")

### B3 — Observe distance collapse as dimensions increase

The curse of dimensionality is not just an abstract concept — you can measure it directly. Fill in the array dimensions and loop bounds to complete the distance function, then observe how mean pairwise distance changes as dimensions increase.

In [ ]:
# B3 — Observe distance collapse as dimensions increase
#
# In high-dimensional spaces, all points tend to become equally far apart,
# making it hard for distance-based algorithms (KNN, clustering) to find
# meaningful neighborhoods.

def mean_pairwise_distance(n_samples, n_dims):
    """Return the mean Euclidean distance between all pairs of random points."""
    np.random.seed(42)
    data = np.random.randn(___, ___)   # YOUR CODE — n_samples rows, n_dims columns

    distances = []
    for i in range(___):              # YOUR CODE — iterate over sample indices
        for j in range(i + 1, ___):  # YOUR CODE — only upper triangle (avoid duplicates)
            dist = np.linalg.norm(data[i] - data[j])
            distances.append(dist)

    return np.mean(distances)


# Compute mean distance at several dimensionalities
n_samples = 50
dimensions = [1, 5, 10, 25, 50, 100]

print(f"{'Dims':>6}  {'Mean pairwise distance':>24}")
for d in dimensions:
    avg = mean_pairwise_distance(n_samples, d)
    print(f"{d:>6}  {avg:>24.4f}")

# --- checks ---
dist_1d   = mean_pairwise_distance(50, 1)
dist_100d = mean_pairwise_distance(50, 100)

assert dist_100d > dist_1d, "Mean pairwise distance should grow with dimensions — check your array shape and loop range"
assert mean_pairwise_distance(50, 1) < 2.0, "At 1 dimension, mean distance should be less than 2.0 — check the array shape: (n_samples, n_dims)"
print("\u2713 B3 complete!")

---
## Section C — Applied Problem

The goal is to write a function that ranks features by how strongly they predict a continuous target. This is a common first step in any ML pipeline — before fitting a model, understanding which features carry signal and which are noise.

In [ ]:
# Section C — Feature signal ranking
#
# Complete the function below. It should return a DataFrame with two columns:
#   'feature'          — the feature name (string)
#   'abs_correlation'  — the absolute value of its Pearson correlation with the target
#
# Rows must be sorted from highest to lowest abs_correlation.

def rank_features(X, y):
    """
    Parameters
    ----------
    X : DataFrame of feature values
    y : Series of target values

    Returns
    -------
    DataFrame with columns ['feature', 'abs_correlation'],
    sorted by abs_correlation descending.
    """
    # YOUR CODE HERE
    pass


# --- Run it ---
X, y = load_california_housing()
ranking = rank_features(X, y)
print(ranking.to_string(index=False))

# --- Plot ---
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(ranking['feature'], ranking['abs_correlation'])
ax.invert_yaxis()
ax.set_xlabel("Absolute Pearson Correlation with Target")
ax.set_title("Feature Signal Strength \u2014 California Housing")
plt.tight_layout()
plt.show()

# --- Section C checks ---
assert isinstance(ranking, pd.DataFrame), "rank_features should return a DataFrame"
assert list(ranking.columns) == ['feature', 'abs_correlation'], "DataFrame must have exactly two columns: 'feature' and 'abs_correlation'"
assert ranking.iloc[0]['feature'] == 'MedInc', "The top feature should be 'MedInc' — sort by abs_correlation descending"
assert ranking['abs_correlation'].is_monotonic_decreasing, "Sort the DataFrame by abs_correlation from highest to lowest"
assert (ranking['abs_correlation'] >= 0).all(), "abs_correlation values must be non-negative — use abs() or .abs()"
print("\u2713 Section C complete!")
print(f"  Top predictor: {ranking.iloc[0]['feature']} (|r| = {ranking.iloc[0]['abs_correlation']:.3f})")

---

## Section D — Reflection

These questions are for reflection. Edit the markdown cells below each question to write your response. There are no wrong answers — we are looking for thoughtful engagement with what you have learned. Your instructor may review these.

**Question D1**

In `working-sessions/ml_concepts/09_the_no_free_lunch_theorem.ipynb`, the same three algorithms ranked differently depending on the shape of the dataset. What does that tell you about how you should choose a model in practice?

*Your response here...*

**Question D2**

You run tree-based feature importance on a customer dataset and find that a column called `transaction_id` ranks as the third most important feature. What would you suspect, and what would you do next?

*Your response here...*

**Question D3**

A colleague argues that adding more features always improves a KNN model because the model has more information. Based on what you observed in B3, how would you respond?

*Your response here...*